# Project 04: Python Modular ETL Frameworks - Educational Walkthrough

Welcome to **Project 04** of the **Data Engineering Diaries** series by Sathiya Murthy.

## Notebook Objectives
In this hands-on notebook, we demonstrate the architectural patterns for building production-grade Python ETL pipelines:
1. **Extraction**: Modular ingestion from CSV, nested JSON, and REST API endpoints.
2. **Transformation**: Header normalization, null handling, metric enrichment, domain validations, and aggregation rollups.
3. **Loading**: Persistent storage into columnar Parquet files, CSV artifacts, and relational SQLite database warehouses.
4. **Orchestration**: End-to-end execution monitoring, configuration management, and dead-letter queue (DLQ) quarantining.

In [ ]:
import sys
from pathlib import Path

# Add project root directory to python path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from src.extractors import CSVExtractor, JSONExtractor, APIExtractor
from src.transformers import DataCleaner, DataEnricher, DataValidator, DataAggregator
from src.loaders import CSVLoader, ParquetLoader, DatabaseLoader
from src.pipeline import PipelineConfig, ETLPipeline, setup_logger

print("Imports successfully loaded!")


## Section 1: Extraction from Heterogeneous Sources

In enterprise environments, data resides across diverse formats. Below we initialize concrete extractor classes to load CSV customers, nested JSON orders, and mock currency exchange rates.

In [ ]:
# Extract CSV Customers
csv_ext = CSVExtractor({"path": "../sample-data/customers.csv"})
df_customers = csv_ext.extract()
print("--- Extracted Customers (CSV) ---")
display(df_customers.head(3))

# Extract JSON Orders
json_ext = JSONExtractor({"path": "../sample-data/orders.json"})
df_orders = json_ext.extract()
print("--- Extracted Orders (JSON) ---")
display(df_orders.head(3))

# Extract API Exchange Rates (Mock)
api_ext = APIExtractor({
    "mock_response": {"base": "USD", "rates": {"USD": 1.0, "EUR": 0.92, "GBP": 0.79}}
})
df_fx = api_ext.extract()
print("--- Extracted Exchange Rates (API) ---")
display(df_fx)


## Section 2: Data Cleaning, Enrichment, & Quality Validation

Next, we process raw DataFrames through a series of modular transformers:
- **DataCleaner**: Trims text strings, forces `snake_case` headers, fills missing values.
- **DataEnricher**: Calculates tax, subtotals, and customer loyalty tier classifications.
- **DataValidator**: Filters invalid emails or negative amounts and routes bad records to a DLQ JSON file.
- **DataAggregator**: Summarizes KPI rollups by customer country.

In [ ]:
# 1. Clean Customers
cleaner = DataCleaner({"lowercase_headers": True, "trim_strings": True, "default_fill_values": {"total_spent": 0.0}})
df_clean_cust = cleaner.transform(df_customers)

# 2. Enrich Customers
enricher = DataEnricher({"customer_tier_thresholds": {"vip": 5000.0, "gold": 1000.0, "silver": 200.0}})
df_enriched_cust = enricher.transform(df_clean_cust)

# 3. Validate Customers
validator = DataValidator({"dlq_path": "../output/notebook_dlq.json"})
df_valid_cust = validator.transform(df_enriched_cust)
print("--- Transformed & Validated Customers ---")
display(df_valid_cust.head(3))

# 4. Aggregations by Country
aggregator = DataAggregator({"group_by_columns": ["country"], "aggregations": {"total_spent": ["sum", "mean", "count"]}})
df_agg_cust = aggregator.transform(df_valid_cust)
print("--- Aggregated Country Metrics ---")
display(df_agg_cust)


## Section 3: Target Persistence & Storage Drivers

We save our cleaned, enriched, and aggregated datasets into three target systems:
1. **ParquetLoader**: Columnar Parquet format with snappy compression.
2. **CSVLoader**: Flat CSV file storage.
3. **DatabaseLoader**: Relational SQLite tables via SQLAlchemy transactions.

In [ ]:
# Save Customers to Parquet
p_loader = ParquetLoader({"path": "../output/notebook_customers.parquet", "compression": "snappy"})
p_loader.load(df_valid_cust)

# Save Summary to CSV
c_loader = CSVLoader({"path": "../output/notebook_summary.csv", "index": False})
c_loader.load(df_agg_cust)

# Save to SQLite Warehouse
db_loader = DatabaseLoader({
    "connection_string": "sqlite:///../output/notebook_warehouse.db",
    "table_name": "dim_customers",
    "if_exists": "replace"
})
db_loader.load(df_valid_cust)

print("All targets successfully written!")


## Section 4: End-to-End Orchestration & Metrics Reporting

Finally, we instantiate `PipelineConfig` using `configs/pipeline_config.yaml` and execute the full automated `ETLPipeline` workflow.

In [ ]:
# Load YAML configuration
config = PipelineConfig("../configs/pipeline_config.yaml")

# Run Pipeline
pipeline = ETLPipeline(config)
metrics = pipeline.run()

# Display summary metrics
import json
print("=== Pipeline Execution Report ===")
print(json.dumps(metrics, indent=2))
